# Clustering a posterior ensemble and reading the clusters with motifs — Boulia ensemble

The steps of `cluster_workflow_synthetic.ipynb`, applied to post-burn-in models of the trans-dimensional (trans-D) inversion of the Boulia area:

1. **Data**: models from five chains.
2. **Features**: each model $m_i$ ($i = 1, \dots, N$) is described by $\mathbf{f}_i \in \mathbb{R}^{523}$: 512 encoder values, 6 topological scalars and 5 density moments.
3. **Embedding and clustering**: UMAP (Uniform Manifold Approximation and Projection; McInnes et al., 2018), then DBSCAN (Density-Based Spatial Clustering of Applications with Noise; Ester et al., 1996).
4. **Cluster statistics**: median and standard-deviation models of each cluster, interactive viewer.
5. **Motifs**: motif enrichment and topology of each cluster.
6. **Hypotheses**: examples of the validated motifs, posterior probabilities of geological hypotheses.

Code cells are numbered by section (`# 3.2`) and run from top to bottom; parameters are in capitals. In Colab, cell 0.1 (folded) fetches the modules and installs the missing packages.

### 0.1  Setup

Fetches the modules (Colab), installs the missing packages and imports them. Run once; folded.

In [ ]:
# @title 0.1  Setup: modules, packages and imports (run once) { display-mode: "form" }
import sys, os
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/jgeomos/PostTopo.git'   
REPO_SUBDIR = ''                                     # folder of the notebook inside the repository ('' = root)
if IN_COLAB:                                         # clone the repository and install what Colab lacks
    repo_dir = '/content/' + REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')
    if not os.path.isdir(repo_dir):
        !git clone -q --depth 1 {REPO_URL} {repo_dir}
    %cd {os.path.join(repo_dir, REPO_SUBDIR)}
    %pip install -q umap-learn connected-components-3d colorcet "bokeh>=3.4"

MODULE_DIR, DATA_DIR = os.path.abspath('modules'), os.path.abspath('data')
assert os.path.isdir(MODULE_DIR), f"{MODULE_DIR} not found: check REPO_URL and REPO_SUBDIR"
os.makedirs(DATA_DIR, exist_ok=True)
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

from importlib import reload
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
import synthetic_ensemble as se, ensemble_features as ef, cluster_tools as ct, cluster_viewer as cv
import motif_analysis as ma, motif_plots as mp, cluster_motifs as cm, course_helpers as ch
for m_ in (se, ef, ct, cv, ma, mp, cm, ch):
    reload(m_)                                       # picks up edits of the modules when the cell is run again
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)
plt.rcParams['figure.dpi'] = 110
print(f"Colab: {IN_COLAB} | working directory: {os.getcwd()}")

## 1. Data

The models were saved by `export_boulia_subset_cell.py`: the same number of models from each chain, evenly spaced after burn-in, lateral padding removed. `core` holds them as $(n_z, n_y, n_x, N)$, depth index 0 at the top. `MAX_NUM_MODELS` keeps an evenly spread subset, e.g. for a quick first run.

### 1.1  Boulia ensemble

Post-burn-in models of five chains, lateral padding removed.

In [ ]:
FIELD_FILE = os.path.join(DATA_DIR, 'boulia_5chains_2000models.npz')
MAX_NUM_MODELS = None    # cap on the number of models, evenly spread over the ensemble (None = all)
FILE_PREFIX = 'boulia_5chains'   # prefix of the files written by the notebook

core, chain_id, padd = ch.load_field(FIELD_FILE, MAX_NUM_MODELS)
nz, ny, nx, N = core.shape

## 2. Features

The encoder has no trained weight: fixed $3 \times 3 \times 3$ difference kernels, absolute value and max pooling, three times, then fixed random projections (Johnson & Lindenstrauss, 1984). Its 512 values describe where the unit contacts lie and their density contrast, at three scales. All 523 features are standardised column by column. They are computed once (about 0.1 s per model and per core), saved in `data/` and reloaded when the notebook runs again on the same models.

### 2.1  Features

Loaded from `data/` if computed before for the same models, computed and saved otherwise.

In [ ]:
RECOMPUTE_FEATURES = False   # True: compute them again (e.g. after changing the encoder)

features, topo = ch.ensemble_features(core, cache=os.path.join(DATA_DIR, FILE_PREFIX), recompute=RECOMPUTE_FEATURES)

## 3. Embedding and clustering

With few neighbours, UMAP keeps the local structure: models differing by small changes form thin strings. DBSCAN links points closer than `EPS` (embedding units); a point with at least `MIN_SAMPLES` neighbours is a core point, and points reached from no core point are noise (label $-1$). `EPS` depends on $N$ and on the UMAP settings: it is chosen near the knee of the $k$-distance curve of cell 3.2. HDBSCAN (Campello et al., 2013) needs no `EPS` but tends to merge the strings.

Cell 3.3 lists the clusters by size: chain shares, mean voxel standard deviation (spread, in density units) and representative model (closest to the centroid).

### 3.1  UMAP embedding

<font color="red"><b>Parameters to change:</b></font>
<ul>
<li><code>UMAP_N_NEIGHBORS</code> (2–50): small keeps thin strings of similar models, large gives broad blobs</li>
<li><code>UMAP_MIN_DIST</code> (0–0.5): small packs similar models tightly, large spreads them out</li>
<li><code>UMAP_METRIC</code>: <code>'cosine'</code>, <code>'euclidean'</code>, <code>'correlation'</code> or <code>'manhattan'</code></li>
</ul>
Then re-run the cells below.


In [ ]:
UMAP_N_NEIGHBORS = 5     # small (2-5): local structure, thin strings of similar models; large (15-50): broad blobs
UMAP_MIN_DIST = 0.05     # small (0-0.05): tight packing, good for clustering; large (0.1-0.5): easier to read
UMAP_METRIC = 'cosine'   # 'cosine' (angle between vectors), 'euclidean', 'correlation' (centred cosine), 'manhattan'

coords = ch.embed(features, UMAP_N_NEIGHBORS, UMAP_MIN_DIST, UMAP_METRIC, chain_id, prefix=FILE_PREFIX)

### 3.2  Clustering of the embedding

<font color="red"><b>Parameters to change:</b></font>
<ul>
<li><code>CLUSTERER</code>: <code>'dbscan'</code> or <code>'hdbscan'</code></li>
<li><code>EPS</code>: DBSCAN radius in embedding units; smaller gives more clusters and more noise (choose it near the knee of the k-distance curve)</li>
<li><code>MIN_SAMPLES</code> (3–20): larger leaves more models as noise</li>
<li><code>MIN_CLUSTER_SIZE</code>: HDBSCAN only, smallest cluster kept</li>
</ul>
Then re-run the cells below.

In [ ]:
CLUSTERER = 'dbscan'     # 'dbscan' (density threshold EPS, tuned per ensemble) or 'hdbscan' (no EPS)
EPS = 0.45               # DBSCAN: neighbourhood radius in embedding units; smaller: more clusters and more noise
MIN_SAMPLES = 5          # neighbours of a core point; small (3-5) keeps small clusters, large (10-20) leaves more noise
MIN_CLUSTER_SIZE = 10    # HDBSCAN only: smallest group of models accepted as a cluster

labels = ch.cluster(coords, CLUSTERER, EPS, MIN_SAMPLES, MIN_CLUSTER_SIZE, prefix=FILE_PREFIX)

### 3.3  Cluster table

Clusters sorted by size (commented out: uncomment to show).

In [ ]:
# display(ct.cluster_table(core, coords, labels, chain_id, min_size=MIN_SAMPLES).round(2).head(20))

## 4. Cluster statistics and viewer

Median and standard-deviation models are computed cell by cell over the members of each cluster. The section at $y$ = `IY` is an $x$–$z$ panel, at $x$ = `IX` a $y$–$z$ panel, the map at $z$ = `IZ` an $x$–$y$ panel. Axes are labelled in cells, or in physical units when `AXIS_COORDS` holds the cell-centre coordinates.

The viewer (4.3) is a self-contained HTML page: hovering over a point shows its model; tapping it, or choosing in the menu, shows its cluster. Clusters smaller than `MIN_SAMPLES` are grey.

### 4.1  Median and standard-deviation models of the clusters

One tile per cluster, on the section at y = `IY`, or each cluster on its section with most structure (`'best'`).

In [ ]:
IY, IX, IZ = ny // 2, nx // 2, 10      # section indices: y = IY (x-z panel), x = IX (y-z panel), map at z = IZ
AXIS_COORDS = None                     # or dict(z=..., y=..., x=...): cell-centre coordinates along the axes of core
AXIS_UNITS = 'cell'                    # unit of AXIS_COORDS, e.g. 'm'
VMIN, VMAX = ch.density_range(core)    # common colour scale of the densities (0.5th and 99.5th percentiles)

ch.plot_tiles(core, labels, 'y', IY, min_size=MIN_SAMPLES, vmin=VMIN, vmax=VMAX,
              axis_coords=AXIS_COORDS, units=AXIS_UNITS)

### 4.2  Largest clusters: median, standard deviation and representative model

Section y = `IY` and map z = `IZ` for the `N_SHOW` largest clusters.

<font color="red">
<b>Parameters to change:</b> <code>N_SHOW</code>

In [ ]:
N_SHOW = 2
ch.plot_largest_clusters(core, coords, labels, N_SHOW, IY, IX, IZ, min_size=MIN_SAMPLES, vmin=VMIN, vmax=VMAX,
                         axis_coords=AXIS_COORDS, units=AXIS_UNITS)

### 4.3  Interactive viewer

Self-contained HTML page, also saved as a file.

In [ ]:
layout = cv.build_viewer(core, coords, labels, chain_id, iy=IY, ix=IX, iz=IZ, min_size=MIN_SAMPLES,
                         vmin=VMIN, vmax=VMAX, title=f'{FILE_PREFIX}: clusters ({CLUSTERER})')
ch.show_viewer(layout, f'{FILE_PREFIX}_cluster_viewer_strings.html', height=1150)

## 5. Motifs and topology of the clusters

Each model $m_i$ is turned into a body graph (bodies: 6-connected sets of cells of one unit, linked by their contacts; Thiele et al., 2016a) and into motif counts $c_k(m_i)$, the number of its bodies realising motif $k$. For a cluster $c$ of $|c|$ models,

$$P(k \mid c) = \frac{1}{|c|}\sum_{i \in c}\mathbf{1}\{c_k(m_i) > 0\}, \qquad P(k) = \frac{1}{N}\sum_{i=1}^{N}\mathbf{1}\{c_k(m_i) > 0\}, \qquad E_{kc} = \frac{P(k \mid c)}{P(k)},$$

where $\mathbf{1}\{\cdot\}$ is 1 when the condition holds and 0 otherwise, $P(k \mid c)$ and $P(k)$ are the fractions of models holding motif $k$ in the cluster and in the ensemble (noise included), and $E_{kc}$ is the enrichment (Milo et al., 2002): $E_{kc} \gg 1$ marks the motifs that characterise the cluster, $E_{kc} \approx 0$ those it excludes. The clusters come from the features, not from the motifs, so that the enrichment is an independent reading of them. Topology classes (unit contacts and bodies per unit; Thiele et al., 2016b) are numbered by decreasing frequency.

Cell 5.1 takes the settings of the motif workflow (`motif_workflow_boulia_v4.ipynb`). Units are matched across models by overlap with the prior model of the sampler (label switching; Stephens, 2000), read from `PRIOR_FILE` and oriented on the grid of `core` automatically. If the check figure of cell 5.2 shows a mirror image of the models, `PRIOR_ROW` selects another row of the orientation table. Without the file, a modal stand-in is built from the ensemble.

Cell 5.3 lists $P(k \mid c)$ and the most frequent topology class of each cluster; the motif viewer (5.4) adds the motifs of each model, colouring by motif, and a heatmap of $E_{kc}$ (tap a row to select the cluster).

### 5.1  Motif settings


In [ ]:
VMIN_VOXELS = 52          # volume floor: bodies with fewer cells are fragments (sampler noise)
MAX_BODIES = 5           # topology key: bodies per unit counted up to this value
ORDER = None             # stratigraphic order of the reference units, top -> bottom (None: by mean depth)
MIN_UNIT_SHARE = 0.01    # reference units below this share of the grid are absorbed by their nearest unit
CELL_SIZES = (1.0, 1.0, 1.0)   # (dz, dy, dx) in m along the axes of core (y and x swapped on loading); placeholder:
                               # shape classes, steepness and layer aspect are then computed in cells
THR = dict(amin_violation=100)           # thresholds overriding ma.DEFAULTS, e.g. dict(c_min=1.5, amin_link=4)
UNIT_NAMES = {}          # names of the reference units, e.g. {0: 'cover'} (0 = top unit)
VIEW_MOTIFS = [m for m in ma.MOTIFS if m != 'fragment']    # motifs in the tables and the viewer
PRIOR_FILE = os.path.join(DATA_DIR, 'boulia_prior_model.npy')   # prior model of the sampler
PRIOR_ROW = 0            # row of the orientation table used for the prior (0 = best agreement)

### 5.2  Reference model and motif table

One row of motif counts per model.

In [ ]:
ref = ch.reference(core, PRIOR_FILE, padd, MIN_UNIT_SHARE, row=PRIOR_ROW, check=(IY, IZ))
K = int(ref.max()) + 1                 # number of reference units; code K - 1 is the basement
kw = dict(vmin_voxels=VMIN_VOXELS, reference=ref, cell_sizes=CELL_SIZES, max_bodies=MAX_BODIES, order=ORDER, thr=THR)
vw = mp.Viewer(core, ref, unit_names=UNIT_NAMES, cell_sizes=CELL_SIZES)      # sections with units matched to ref

df_motifs = ma.motif_table(core, n_jobs=-1, **kw)
display(ma.motif_probabilities(df_motifs).T.round(3))

### 5.3  Motifs per cluster

P(motif | cluster) and most frequent topology class of each cluster.

In [ ]:
topo_class = ch.motifs_per_cluster(df_motifs, labels, VIEW_MOTIFS, min_size=MIN_SAMPLES)

### 5.4  Motif viewer

The first viewer plus the motifs of each model, colouring by motif and the enrichment heatmap.

In [ ]:
VIEW_SECTION = 'y'       # section in the hover and the first panels: 'y' (at y = IY) or 'x' (at x = IX)

layout_m = cv.build_viewer(core, coords, labels, chain_id, iy=IY, ix=IX, iz=IZ, min_size=MIN_SAMPLES,
                           vmin=VMIN, vmax=VMAX, thumb_orient=VIEW_SECTION, orient=VIEW_SECTION,
                           motifs=df_motifs[VIEW_MOTIFS], topology=topo_class,
                           extra=ch.viewer_extra(df_motifs, topo), tooltip_extra=['n_units', 'n_born'],
                           title=f'{FILE_PREFIX}: clusters ({CLUSTERER}), motifs and topology')
ch.show_viewer(layout_m, f'{FILE_PREFIX}_cluster_viewer_motifs.html', height=1600)

## 6. Looking at the motifs

The motifs shown were validated on the synthetic ensemble (`cluster_workflow_synthetic.ipynb`, section 6): `chimney` (pipe linking basement and cover), `dyke` (steep sheet cutting several units), `window` (basement breaching the cover) and `enclosed_body` (lens or enclave; a lower bound on lenses). Born bodies of the Boulia ensemble are less regular, and their shape classes depend on `CELL_SIZES` and `THR`.

The examples of cells 6.2 and 6.3 are the clearest realisations of each motif among up to 200 checked models per motif. Their clarity is $s = \min_j \mu_j$, where $\mu_j \in [0, 1]$ is the margin of criterion $j$, 0 at the threshold of the definition and 1 when fully typical. The criteria are those of the typicality score, plus those of the geological reading that the definition does not test: the plunge of a `chimney` (the definition accepts flat rods between basement and cover) and a localised `window` (the definition accepts a basement lying under the cover everywhere). Bodies carrying no other motif, and models with fewer flagged bodies, are preferred. Clarity selects what is shown; it does not change the probabilities.

Each hypothesis $H$ is an indicator $\mathbf{1}_H(m)$ (1 when model $m$ satisfies $H$, 0 otherwise) evaluated on the motif table, and

$$P(H \mid d) \approx \frac{1}{N_s}\sum_{i=1}^{N_s} \mathbf{1}_H(m_i),$$

where $d$ denotes the data and $N_s$ the number of selected models (the ensemble or one cluster). The bars of cell 6.4 show the binomial standard error $\sqrt{P(1 - P)/N_s}$, with $P = P(H \mid d)$, valid for independent samples.

| Hypothesis | Indicator |
|---|---|
| at least one unit born | `n_born` $> 0$ (units not matched to a reference unit) |
| basement is one body | the deepest reference unit has a single body |
| lens / enclosed body present | `enclosed_body` $> 0$ |
| basement breaches the cover | `window` $> 0$ |
| pipe from basement into cover | `chimney` $> 0$ |
| steep dyke cutting several units | `dyke` $> 0$ (not `cross_cutting`, which also counts pipes) |
| basement within `DEPTH_CELLS` cells of the surface | `z_deep_top` $<$ `DEPTH_CELLS` (depth index of the shallowest basement cell) |

### 6.1  Parameters to play with

Re-run 6.1 to 6.4 after a change.

<font color="red"><b>Parameters to change:</b></font>
<ul>
<li><code>SHOW_MOTIFS</code> (list) and <code>EXAMPLE_MOTIF</code> (one motif), among:
  <ul>
  <li><code>'chimney'</code>: pipe linking basement and cover</li>
  <li><code>'dyke'</code>: steep sheet cutting several units</li>
  <li><code>'window'</code>: basement breaching the cover</li>
  <li><code>'enclosed_body'</code>: lens, enclave</li>
  <li><code>'layer'</code>: laterally continuous sheet</li>
  <li><code>'steep_body'</code>, <code>'cross_cutting'</code>: steep bodies (dyke or pipe)</li>
  <li><code>'order_violation'</code>: inverted contact</li>
  <li><code>'bridge'</code>: separator</li>
  </ul>
</li>
<li><code>CLUSTER</code>: <code>None</code> (whole ensemble) or a cluster number from the viewer</li>
<li><code>N_EX</code>: number of examples per motif in the catalogue (6.3)</li>
<li><code>GROUP_BY</code>: <code>'chain'</code> or <code>'cluster'</code></li>
</ul>


In [ ]:
# SHOW_MOTIFS = list(cm.VALIDATED_MOTIFS)   # chimney, dyke, window, enclosed_body
SHOW_MOTIFS = ['window', 'enclosed_body', 'order_violation', 'chimney', 'dyke', 'layer', 'steep_body']
CLUSTER = None              # None: whole ensemble; or a cluster number read in the viewer, e.g. 7
N_EX = 5                    # example models per motif in the catalogue (6.3)
GROUP_BY = 'chain'          # probabilities per 'chain', or per 'cluster' (the N_GROUPS largest clusters)
N_GROUPS = 6
EXAMPLE_MOTIF = 'window'
DEPTH_CELLS = 3             # hypothesis 'basement within DEPTH_CELLS cells of the surface'
MY_HYPOTHESES = {}          # your own, as functions of the motif table d, e.g.
                            # {'two born units or more': lambda d: d['n_born'] >= 2}

members, where = ch.selection(df_motifs, core, kw, labels, CLUSTER, SHOW_MOTIFS + [EXAMPLE_MOTIF])

### 6.2  Clearest example of a motif

Criteria, bodies and sections of the clearest example of `EXAMPLE_MOTIF` (model numbers as in the viewer's hover).

### 6.1  Parameter to play with

Re-run 6.1 to 6.4 after a change.

<font color="red"><b>Parameters to change:</b></font>
<ul>
<li> <code>EXAMPLE_MOTIF</code> (one motif):
  <ul>
  <li><code>'chimney'</code>: pipe linking basement and cover</li>
  <li><code>'dyke'</code>: steep sheet cutting several units</li>
  <li><code>'window'</code>: basement breaching the cover</li>
  <li><code>'enclosed_body'</code>: lens, enclave</li>
  <li><code>'layer'</code>: laterally continuous sheet</li>
  <li><code>'steep_body'</code>, <code>'cross_cutting'</code>: steep bodies (dyke or pipe)</li>
  <li><code>'order_violation'</code>: inverted contact</li>
  <li><code>'bridge'</code>: separator</li>
  </ul>
</li>
</ul>


In [ ]:
EXAMPLE_MOTIF = 'window'   # motif of the example model (6.2)
examples = ch.rank_examples(core, df_motifs, kw, SHOW_MOTIFS + [EXAMPLE_MOTIF], members)   # clearest first
ch.show_example(vw, kw, examples, EXAMPLE_MOTIF, labels, chain_id)

### 6.3  Catalogue of examples

The `N_EX` clearest examples of each motif of `SHOW_MOTIFS`, the body outlined (window: basement cells under the cover).

<font color="red"><b>Parameters to change:</b></font>
<ul>
<li> <code>SHOW_MOTIFS</code> (one motif or more):
  <ul>
  <li><code>'chimney'</code>: pipe linking basement and cover</li>
  <li><code>'dyke'</code>: steep sheet cutting several units</li>
  <li><code>'window'</code>: basement breaching the cover</li>
  <li><code>'enclosed_body'</code>: lens, enclave</li>
  <li><code>'layer'</code>: laterally continuous sheet</li>
  <li><code>'steep_body'</code>, <code>'cross_cutting'</code>: steep bodies (dyke or pipe)</li>
  <li><code>'order_violation'</code>: inverted contact</li>
  <li><code>'bridge'</code>: separator</li>
  </ul>
</li>
</ul>

In [ ]:
SHOW_MOTIFS = ['window','enclosed_body']

ch.plot_catalogue(vw, df_motifs, kw, examples, SHOW_MOTIFS, N_EX, members, where)

### 6.4  Probabilities of motifs and hypotheses

P(motif present) in the selection, and P(H | d) per chain or per cluster.

In [ ]:
ch.motif_presence(df_motifs, SHOW_MOTIFS, members, where)
ch.hypothesis_probabilities(df_motifs, members, where, labels, chain_id, basement=K - 1, group_by=GROUP_BY,
                            min_size=MIN_SAMPLES, n_groups=N_GROUPS, depth_cells=DEPTH_CELLS, extra=MY_HYPOTHESES)

### Notes

- Two ensembles are comparable in feature space only if encoded with the same density range (`vmin`, `vmax` of `DiffNetEncoder.encode_ensemble`). UMAP embeddings change between UMAP versions, and `EPS` is chosen anew for each ensemble.
- In a cluster of 10 models, an enrichment of 2 for a motif present in 20 % of the ensemble rests on 4 models.
- Motif probabilities are conditional on `THR`, `VMIN_VOXELS` and `CELL_SIZES`, which are reported with them.
- With `CELL_SIZES = (1, 1, 1)`, shape classes and steepness are computed in cells, not in metres: the Boulia mesh is set before reporting them.
- The cluster labels are saved as `<FILE_PREFIX>_cluster_labels.npy`, for the motif workflow on the same models in the same order (`chain_id` and `index_in_chain` of the `.npz` identify each model).

### References

- Campello, R.J.G.B., Moulavi, D. & Sander, J., 2013. Density-based clustering based on hierarchical density estimates, *PAKDD*, LNCS 7819, 160–172.
- Ester, M., Kriegel, H.-P., Sander, J. & Xu, X., 1996. A density-based algorithm for discovering clusters in large spatial databases with noise, *Proc. KDD-96*, 226–231.
- Johnson, W.B. & Lindenstrauss, J., 1984. Extensions of Lipschitz mappings into a Hilbert space, *Contemp. Math.*, 26, 189–206.
- McInnes, L., Healy, J. & Melville, J., 2018. UMAP: Uniform Manifold Approximation and Projection for dimension reduction, arXiv:1802.03426.
- Milo, R., Shen-Orr, S., Itzkovitz, S., Kashtan, N., Chklovskii, D. & Alon, U., 2002. Network motifs: simple building blocks of complex networks, *Science*, 298(5594), 824–827.
- Stephens, M., 2000. Dealing with label switching in mixture models, *J. R. Stat. Soc. B*, 62(4), 795–809.
- Thiele, S.T., Jessell, M.W., Lindsay, M., Ogarko, V., Wellmann, J.F. & Pakyuz-Charrier, E., 2016a. The topology of geology 1: Topological analysis, *J. Struct. Geol.*, 91, 27–38.
- Thiele, S.T., Jessell, M.W., Lindsay, M., Wellmann, J.F. & Pakyuz-Charrier, E., 2016b. The topology of geology 2: Topological uncertainty, *J. Struct. Geol.*, 91, 74–87.